In [22]:
import os
import sys
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, DateType, DecimalType, TimestampType
)
from pyspark.sql.functions import col, count, when, isnull, lower, trim, to_date, row_number, max, min, upper

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

spark = (
    SparkSession.builder
    .appName("Lesson21-DataFrames-SparkSQL")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "matrix")
    .config("spark.hadoop.fs.s3a.secret.key", "matrix123")
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .getOrCreate()
)


In [23]:
schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("merchant_id", StringType(), True),
    StructField("merchant_category", StringType(), True),
    StructField("amount", DecimalType(10, 2), True),
    StructField("currency", StringType(), True),
    StructField("status", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("transaction_ts", TimestampType(), True),
    StructField("updated_at", TimestampType(), True)
])

df_raw = spark.read.csv("/opt/data/transactions.csv", header=True, schema=schema)

df_raw.show(5, truncate=False)

+--------------+-----------+-----------+-----------------+------+--------+-------+-------+-------------------+-------------------+
|transaction_id|customer_id|merchant_id|merchant_category|amount|currency|status |channel|transaction_ts     |updated_at         |
+--------------+-----------+-----------+-----------------+------+--------+-------+-------+-------------------+-------------------+
|TXN0007783    |C0252      |M060       |Pharmacy         |19.60 |AZN     |SUCCESS|MOBILE |2026-10-07 21:10:57|2026-10-07 21:10:59|
|TXN0003875    |C1094      |M035       |Utilities        |96.46 |AZN     |SUCCESS|POS    |2026-10-07 13:44:16|2026-10-07 13:44:19|
|TXN0004850    |C0172      |M076       |Fuel             |31.40 |USD     |SUCCESS|WEB    |2026-10-07 13:36:03|2026-10-07 13:36:04|
|TXN0013013    |C0255      |M030       |Grocery          |26.03 |AZN     |SUCCESS|MOBILE |2026-10-07 22:35:12|2026-10-07 22:35:13|
|TXN0014848    |C0210      |M005       |Grocery          |28.69 |USD     |SUCCESS|M

### Her sutundaki null deyer sayisi

In [24]:
null_counts = df_raw.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns])
print("Null counts for each column:")
null_counts.show()

Null counts for each column:
+--------------+-----------+-----------+-----------------+------+--------+------+-------+--------------+----------+
|transaction_id|customer_id|merchant_id|merchant_category|amount|currency|status|channel|transaction_ts|updated_at|
+--------------+-----------+-----------+-----------------+------+--------+------+-------+--------------+----------+
|             0|          0|          0|                0|   240|       0|     0|      0|             0|         0|
+--------------+-----------+-----------+-----------------+------+--------+------+-------+--------------+----------+



### Amount sutunundaki menfi deyer sayisi

In [25]:
negative_amounts_count = df_raw.filter(F.col("amount") < 0).count()
print(f"Number of transactions with negative amounts: {negative_amounts_count}")

Number of transactions with negative amounts: 300


### Tekrarlanan tranzaksiya_id sayisi

In [26]:
dublicate_transactions_count = df_raw.groupBy("transaction_id").count().filter(F.col("count") > 1).count()
print(f"Number of duplicate transactions: {dublicate_transactions_count}")

Number of duplicate transactions: 600


### String deyerlerin ferqli yazilisi

In [27]:
print("Different string values in 'status' column:")
df_raw.select("status").distinct().show()

Different string values in 'status' column:
+---------+
|   status|
+---------+
|  success|
| Reversed|
|   failed|
| SUCCESS |
| REVERSED|
|  Success|
|  SUCCESS|
|   FAILED|
|REVERSED |
|   Failed|
|  pending|
|  Pending|
| reversed|
|  PENDING|
| PENDING |
|  FAILED |
+---------+



In [28]:
print("Different string values in 'merchant_category' column:")
df_raw.select("merchant_category").distinct().show()

Different string values in 'merchant_category' column:
+-----------------+
|merchant_category|
+-----------------+
|           travel|
|        utilities|
|      Restaurant |
|     Electronics |
|     Electronics |
|          Travel |
|        Pharmacy |
|            Fuel |
|      Restaurant |
|           Travel|
|        Pharmacy |
|        UTILITIES|
|          Grocery|
|      Electronics|
|             fuel|
|         pharmacy|
|        Utilities|
|      ELECTRONICS|
|         Pharmacy|
|           TRAVEL|
+-----------------+
only showing top 20 rows



In [29]:
print("Different string values in 'currency' column:")
df_raw.select("currency").distinct().show()

Different string values in 'currency' column:
+--------+
|currency|
+--------+
|     usd|
|     azn|
|     EUR|
|     AZN|
|     AZN|
|    USD |
|     eur|
|    AZN |
|     EUR|
|     USD|
|     USD|
|    EUR |
+--------+



## Cleaning

In [30]:
df_cleaned = df_raw.filter(F.col("amount") >= 0)
df_cleaned = df_cleaned.withColumn("status", lower(trim(col("status")))) \
                       .withColumn("currency", upper(trim(col("currency")))) \
                       .withColumn("merchant_category", lower(trim(col("merchant_category"))))
df_cleaned = df_cleaned.withColumn("transaction_date", to_date(col("transaction_ts")))
df_cleaned.createOrReplaceTempView("cleaned_transactions")
dedup_query = """
    SELECT *
    FROM (
        SELECT 
            *,
            ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY updated_at DESC) as row_num
        FROM cleaned_transactions
    )
    WHERE row_num = 1
"""

df_dedup = spark.sql(dedup_query).drop("row_num")

initial_count = df_raw.count()
cleaned_count = df_cleaned.count()
dedup_count = df_dedup.count()
print(f"Initial count: {initial_count}") 
print(f"Cleaned count: {cleaned_count}")
print(f"Deduplicated count: {dedup_count}")

Initial count: 20737
Cleaned count: 20197
Deduplicated count: 19460


### Loading data into postgres

In [31]:
JDBC_URL = "jdbc:postgresql://postgres:5432/lesson23" 

PG_USER = "matrix"
PG_PASSWORD = "matrix123" 
PG_DRIVER = "org.postgresql.Driver"

df_dedup.write.jdbc(
    url=JDBC_URL,
    table="public.transactions",
    mode="overwrite",
    properties={
        "user": PG_USER, 
        "password": PG_PASSWORD,
        "driver": PG_DRIVER, 
        "batchsize": "5000"
    }
)

print("Data written to PostgreSQL successfully.")

Data written to PostgreSQL successfully.


### En son transaction_date sutunu spark ile tapilmasi

In [32]:
df_pg = spark.read.jdbc(
    url=JDBC_URL,
    table="public.transactions",
    properties={
        "user": PG_USER,
        "password": PG_PASSWORD,
        "driver": PG_DRIVER
    }
)

df_pg.createOrReplaceTempView("pg_transactions")

final_query = """
    SELECT 
        transaction_date AS latest_date,
        COUNT(*) AS total_rows,
        MIN(transaction_id) AS min_id,
        MAX(transaction_id) AS max_id
    FROM pg_transactions
    WHERE transaction_date = (SELECT MAX(transaction_date) FROM pg_transactions)
    GROUP BY transaction_date
"""

df_result = spark.sql(final_query)
df_result.show(truncate=False)

+-----------+----------+----------+----------+
|latest_date|total_rows|min_id    |max_id    |
+-----------+----------+----------+----------+
|2026-10-07 |19460     |TXN0000001|TXN0020000|
+-----------+----------+----------+----------+



### 5-ci tapşırıqda tapdığınız tarixin datasını PostgreSQL-dən 8 paralel hissəyə bölərək oxuyun. 


In [33]:
result_row = df_result.first()
latest_date = result_row["latest_date"]

min_id = int(result_row["min_id"].replace("TXN", ""))
max_id = int(result_row["max_id"].replace("TXN", ""))

print(f"Target date for parallel read: {latest_date} | ID Range: {min_id} to {max_id}")

query = f"""
    (SELECT *, CAST(REPLACE(transaction_id, 'TXN', '') AS BIGINT) AS part_id 
     FROM public.transactions 
     WHERE transaction_date = '{latest_date}') AS subq
"""


df_parallel = spark.read.jdbc(
    url=JDBC_URL,
    table=query,                     
    column="part_id",                
    lowerBound=min_id,               
    upperBound=max_id,               
    numPartitions=8,                 
    properties={
        "user": PG_USER,
        "password": PG_PASSWORD,
        "driver": PG_DRIVER
    }
).drop("part_id") 

actual_partitions = df_parallel.rdd.getNumPartitions()
print(f"Data successfully read in parallel! Partition count: {actual_partitions}")
print(f"Total row count read from PostgreSQL: {df_parallel.count()}")

Target date for parallel read: 2026-10-07 | ID Range: 1 to 20000
Data successfully read in parallel! Partition count: 8
Total row count read from PostgreSQL: 19460


### 7. Paralel oxunmuş datanı MinIO-dakı Delta cədvəlinə transaction_date üzrə partisiyalayaraq yazın. Yazma nəticəsində cədvəldə 200 parquet faylı yaransın. Fayl sayını həm Delta-nın öz məlumatından, həm də MinIO-dakı qovluqdan yoxlayın.


In [ ]:
minio_delta_path = "s3a://mentor.dersi/delta/transactions"

df_parallel.repartition(200) \
    .write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(minio_delta_path)

print("Data successfully written to MinIO Delta Lake!")

df_delta = spark.read.format("delta").load(minio_delta_path)
print(f"Active partition count in Delta: {df_delta.rdd.getNumPartitions()}")

sc = spark.sparkContext
path = sc._gateway.jvm.org.apache.hadoop.fs.Path(minio_delta_path)
fs = path.getFileSystem(sc._jsc.hadoopConfiguration())

files = fs.listFiles(path, True)
parquet_count = 0

while files.hasNext():
    file_name = files.next().getPath().getName()
    if file_name.endswith(".parquet"):
        parquet_count += 1

print(f"Total Parquet files in MinIO directory: {parquet_count}")

Data successfully written to MinIO Delta Lake!
Active partition count in Delta: 40
Total Parquet files in MinIO directory: 200


### 8. Delta cədvəlini optimallaşdırın: müştəri üzrə axtarışları sürətləndirmək üçün məlumatı müştəri identifikatoruna görə yenidən düzün və parquet sayını 10-a yaxın səviyyəyə endirin.

In [ ]:
from delta.tables import DeltaTable

minio_delta_path = "s3a://mentor.dersi/delta/transactions"

deltaTable = DeltaTable.forPath(spark, minio_delta_path)

deltaTable.optimize().executeZOrderBy("customer_id")

print("Delta table successfully optimized with Z-Ordering on customer_id!")

sc = spark.sparkContext
path = sc._gateway.jvm.org.apache.hadoop.fs.Path(minio_delta_path)
fs = path.getFileSystem(sc._jsc.hadoopConfiguration())

files = fs.listFiles(path, True)
parquet_count = 0

while files.hasNext():
    file_name = files.next().getPath().getName()
    if file_name.endswith(".parquet"):
        parquet_count += 1

print(f"Total Parquet files in MinIO folder: {parquet_count}")

Delta table successfully optimized with Z-Ordering on customer_id!
Total Parquet files in MinIO folder: 201


### 9. Delta cədvəlindən oxuyun və kateqoriya üzrə analiz aparın. Yalnız AZN valyutası ilə edilmiş tranzaksiyalar üçün hər kateqoriyada bunları hesablayın: ümumi tranzaksiya sayı, uğurlu tranzaksiyaların sayı, uğursuz tranzaksiyaların faizi, uğurlu tranzaksiyaların ümumi və orta məbləği. Nəticəni uğurlu məbləğin azalan sırası ilə göstərin. Ən yüksək uğursuzluq faizi olan kateqoriyanı müəyyən edin.


In [ ]:
minio_delta_path = "s3a://mentor.dersi/delta/transactions"

df_delta = spark.read.format("delta").load(minio_delta_path)
df_delta.createOrReplaceTempView("delta_transactions")

category_analysis_query = """
    SELECT 
        merchant_category,
        COUNT(*) AS total_transactions,
        COUNT(CASE WHEN status = 'success' THEN 1 END) AS successful_transactions,
        ROUND(
            (COUNT(CASE WHEN status != 'success' THEN 1 END) * 100.0) / COUNT(*), 2
        ) AS failed_percentage,
        ROUND(SUM(CASE WHEN status = 'success' THEN amount ELSE 0 END), 2) AS total_successful_amount,
        ROUND(AVG(CASE WHEN status = 'success' THEN amount ELSE NULL END), 2) AS avg_successful_amount
    FROM delta_transactions
    WHERE currency = 'AZN'
    GROUP BY merchant_category
    ORDER BY total_successful_amount DESC
"""

df_category_analysis = spark.sql(category_analysis_query)

print("Category Analysis for AZN Transactions (Sorted by Total Successful Amount DESC):")
df_category_analysis.show(truncate=False)

highest_failed_category = df_category_analysis.orderBy(col("failed_percentage").desc()).first()

print("--------------------------------------------------------------------------------")
print(f"Highest failure rate category: {highest_failed_category['merchant_category']} "
      f"with {highest_failed_category['failed_percentage']}% failure rate.")
print("--------------------------------------------------------------------------------")

Category Analysis for AZN Transactions (Sorted by Total Successful Amount DESC):
+-----------------+------------------+-----------------------+-----------------+-----------------------+---------------------+
|merchant_category|total_transactions|successful_transactions|failed_percentage|total_successful_amount|avg_successful_amount|
+-----------------+------------------+-----------------------+-----------------+-----------------------+---------------------+
|electronics      |993               |863                    |13.09            |432146.45              |500.75               |
|travel           |968               |826                    |14.67            |300286.95              |363.54               |
|restaurant       |4916              |4197                   |14.63            |152018.41              |36.22                |
|utilities        |2376              |2031                   |14.52            |137569.90              |67.74                |
|grocery          |3288       

### 
10. Delta cədvəlindən oxuyun və saatlıq analiz aparın. AZN valyutası ilə uğurla başa çatmış tranzaksiyalar üçün hər saat üzrə (0–23) tranzaksiya sayını və ümumi məbləği hesablayın. Gün ərzində toplanan (kumulyativ) məbləği də əlavə edin. Ən çox gəlir gətirən 3 saatı göstərin.

In [38]:
minio_delta_path = "s3a://mentor.dersi/delta/transactions"

df_delta = spark.read.format("delta").load(minio_delta_path)
df_delta.createOrReplaceTempView("delta_transactions")

hourly_analysis_query = """
    WITH hourly_stats AS (
        SELECT 
            HOUR(transaction_ts) AS transaction_hour,
            COUNT(*) AS total_transactions,
            ROUND(SUM(amount), 2) AS total_amount
        FROM delta_transactions
        WHERE currency = 'AZN' AND status = 'success'
        GROUP BY HOUR(transaction_ts)
    )
    SELECT 
        transaction_hour,
        total_transactions,
        total_amount,
        ROUND(
            SUM(total_amount) OVER (ORDER BY transaction_hour ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 2
        ) AS cumulative_amount
    FROM hourly_stats
    ORDER BY transaction_hour ASC
"""

df_hourly_analysis = spark.sql(hourly_analysis_query)

print("Hourly Analysis for Successful AZN Transactions (0-23 Hours):")
df_hourly_analysis.show(24, truncate=False)

print("Top 3 Most Profitable Hours:")
df_top3_hours = df_hourly_analysis.orderBy(col("total_amount").desc()).limit(3)
df_top3_hours.show(truncate=False)

Hourly Analysis for Successful AZN Transactions (0-23 Hours):
+----------------+------------------+------------+-----------------+
|transaction_hour|total_transactions|total_amount|cumulative_amount|
+----------------+------------------+------------+-----------------+
|0               |81                |7662.45     |7662.45          |
|1               |88                |8540.69     |16203.14         |
|2               |80                |5254.71     |21457.85         |
|3               |95                |6710.11     |28167.96         |
|4               |97                |8378.22     |36546.18         |
|5               |169               |11194.67    |47740.85         |
|6               |335               |33175.77    |80916.62         |
|7               |647               |62802.96    |143719.58        |
|8               |823               |77514.49    |221234.07        |
|9               |708               |61593.30    |282827.37        |
|10              |709               |5989

In [39]:
spark.stop()